Visual Storyteller: data, model and training

In [10]:
import os
import torchvision
import torch
from PIL import Image
import pandas as pd
from collections import Counter
torch.manual_seed(42)
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.nn.utils.rnn import pad_sequence
import torchvision.models as models
import torch.nn as nn

In [2]:
# make sure captions loaded correctly
df=pd.read_csv('captions.txt')
print(df.head())

                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  


In [3]:
print("total rows:", len(df))

total rows: 40455


In [4]:
#map words to numbers
def get_vocab(df, min_count=2):
    word_counts = Counter()
    for text in df['caption']:
        word_counts.update(text.lower().split())
    
    #special tokens
    mapping = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
    for word, count in word_counts.items():
        if count >= min_count:
            mapping[word] = len(mapping)
    return mapping

word_to_idx = get_vocab(df)
print("total unique words:", len(word_to_idx))

total unique words: 5241


In [5]:
print("unique images:", df['image'].nunique())

unique images: 8091


BELOW IS dataset and dataloader

In [6]:
class FlickrDataset(Dataset):
    def __init__(self, data, img_dir, word_map):
        self.data = data
        self.img_dir = img_dir
        self.word_map = word_map
        
        # resize and normalize using imagenet stats
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'])
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)

        words = row['caption'].lower().split()
        caption_ids = [self.word_map['<START>']]
        caption_ids += [self.word_map.get(w, self.word_map['<UNK>']) for w in words]
        caption_ids += [self.word_map['<END>']]


        return img, torch.tensor(caption_ids)

In [7]:
IMG_DIR="Images"

dataset=FlickrDataset(df, IMG_DIR, word_to_idx)
img, caption_ids=dataset[0]
print(f"image shape:{img.shape}")
print(f"caption length:{len(caption_ids)}")
print(f"first few token id:{caption_ids[:6]}")

image shape:torch.Size([3, 224, 224])
caption length:20
first few token id:tensor([1, 4, 5, 6, 4, 7])


In [8]:
def collate_fn(batch):
    imgs, captions=zip(*batch)
    imgs=torch.stack(imgs)
    captions=pad_sequence(captions, batch_first=True, padding_value=word_to_idx['<PAD>'])
    return imgs, captions

# train/val split
train_size=int(0.9 * len(dataset))
val_size=len(dataset)-train_size
train_data, val_data=torch.utils.data.random_split(dataset, [train_size, val_size])


train_loader=DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader=DataLoader(val_data, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=0)

In [9]:
print(f"train batches:{len(train_loader)}")
print(f"val batches:{len(val_loader)}")

train batches:1138
val batches:127


 Encoder and decoder

In [11]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        resnet = models.resnet18(weights='DEFAULT')
        #remove final classification layer
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        #freeze encoder weights
        for param in self.resnet.parameters():
            param.requires_grad = False
        self.fc = nn.Linear(512, embed_dim)

    def forward(self, imgs):
        with torch.no_grad():
            features = self.resnet(imgs)
        features = features.squeeze(-1).squeeze(-1)
        return self.fc(features)

In [12]:
class DecoderLSTM(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)


    def forward(self, features, captions):
        #image features as initial hidden state
        hidden = features.unsqueeze(0)
        cell = torch.zeros_like(hidden)
        embeddings = self.embedding(captions[:, :-1])
        outputs, _ = self.lstm(embeddings, (hidden, cell))
        return self.fc(outputs)

In [14]:
encoder = EncoderCNN(embed_dim=256)
decoder = DecoderLSTM(embed_dim=256, hidden_dim=512, vocab_size=len(word_to_idx))
print("ok")

ok
